## HW 4

### Описание
В скором времени AI агенты достигнут уровня развития человека. А значит им придется столкнуться с проблемами, которые раньше были свойственны только людям.
Им придется сдавать ЕГЭ. Поэтому мы решили заранее помочь им подготовиться к экзамену по литературе.
Агент прочитал романы Война и Мир Толстого и Преступление и наказание Достоевского, но мало что запомнил. Поможем ему понять эти романы лучше.
Для этого мы научим его определять по вырванной цитате из романа, к какому роману она относится.

### Данные
train.csv - обучающая выборка, содержит 2 поля:
* text - текст цитаты
* label - метка класса (0 - Война и Мир, 1 - Преступление и наказание)

mlm.txt - plain текст вперемешку цитат из обоих романов.

### Требования к решению
* Ноутбук с кодом и метриками
* submission.csv на основе test.csv с предсказаниями залитый на платформу контестов
* accuracy на test: 0.56 и меньше - 0 баллов, 0.75 и больше - 10 баллов

### Подсказки
* Вероятно ваши метрики на валидации (как до MLM так и после) получились сильно выше, чем на test
* Текст в test обрабатывался чуть иначе, чем в train
* Надо понять в чем причина, исправить. Ваши метрики на val станут ниже, зато на test повысятся.

### Задача
1. **Обучить классификатор** (фактически та же задача, что и на практике 5):
* Разбить train.csv на train и valid выборки
* Обучить модель, которая по тексту цитаты будет определять к какому роману она относится. Необходимо использовать модель "distilbert/distilroberta-base" из huggingface.
* Протестировать на valid: confusion_matrix, accuracy, f1. Сохранить эти метрики.




In [1]:
!pip install pytorch-lightning transformers datasets --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 57.0 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import re

def normalize_text(text):
    text = text.replace("—", "-").replace("–", "-")
    text = text.replace("«", '"').replace("»", '"')
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("„", '"')
    text = text.replace("…", "...")
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    return text


In [5]:
import pandas as pd
data = load_dataset(
    "csv",
    data_files={
        "train": "/content/drive/MyDrive/DL_HW4/train.csv"
    })
def normalize_dataset(dataset):
    return dataset.map(lambda x: {"text": normalize_text(x["text"])})

data["train"] = normalize_dataset(data["train"])

data_split = data["train"].train_test_split(test_size=0.2, seed=42)

train_data = data_split["train"]
valid_data = data_split["test"]



Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/5640 [00:00<?, ? examples/s]

In [6]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
import numpy as np


In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilroberta-base")

def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length"
    )


train_data = train_data.map(tokenize_function, batched=True)
valid_data = valid_data.map(tokenize_function, batched=True)

columns_to_set = ["input_ids", "attention_mask", "label"]
train_data.set_format(type="torch", columns=columns_to_set)
valid_data.set_format(type="torch", columns=columns_to_set)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/4512 [00:00<?, ? examples/s]

Map:   0%|          | 0/1128 [00:00<?, ? examples/s]

In [11]:
# model_first = AutoModelForSequenceClassification.from_pretrained("distilbert/distilroberta-base", num_labels=2)

model_first = AutoModelForSequenceClassification.from_pretrained(
    "/content/drive/MyDrive/DL_HW4/model_first_final",
    num_labels=2
)

In [12]:
from transformers import TrainingArguments, Trainer
import numpy as np
import pandas as pd


In [13]:

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    conf_m = confusion_matrix(labels, predictions)
    a_score = accuracy_score(labels, predictions)
    f1_s = f1_score(labels, predictions, average="weighted")
    return {"CONFUSION_MATRIX": conf_m, "ACCURACY": a_score, "F1": f1_s}

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    save_strategy="epoch",
    logging_strategy="epoch",
    push_to_hub=False,
    report_to=[]
)


trainer = Trainer(
    model=model_first,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=valid_data,
    compute_metrics=compute_metrics
)



In [14]:
trainer.train()

results = trainer.evaluate()



Step,Training Loss
564,0.652900
1128,0.621300
1692,0.595000


In [15]:
print('МЕТРИКИ:')
print(f"Accuracy = {results['eval_ACCURACY']}")
print(f"F1 = {results['eval_F1']}")
print('Confusion matrix:')
print(results['eval_CONFUSION_MATRIX'])

МЕТРИКИ:
Accuracy = 0.6728723404255319
F1 = 0.6727726986177459
Confusion matrix:
[[358 189]
 [180 401]]


In [16]:

directory = "/content/drive/MyDrive/DL_HW4/model_first_final2"
trainer.save_model(directory)


In [17]:
import pandas as pd
import torch

data_p = "/content/drive/MyDrive/DL_HW4/submission.csv"
save_p = "/content/drive/MyDrive/DL_HW4/submission_predictions_manual2.csv"

df = pd.read_csv(data_p)
texts = [normalize_text(t) for t in df['text']]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(device)

model_first.to(device).eval()
with torch.no_grad():
    df["label"] = model_first(**inputs).logits.argmax(dim=1).cpu().numpy()

df.to_csv(save_p, index=False)



2. **Претренировать модель с помощью unsupervised masked language modeling** на train-test.txt
* Воспользоваться вот этим туториалом https://huggingface.co/docs/transformers/main/tasks/masked_language_modeling
* Вывести метрики perplexity и loss до и после обучения


In [18]:
!pip install -q transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


In [55]:
import math
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)
from google.colab import drive
import re

with open("/content/drive/MyDrive/DL_HW4/mlm.txt", 'r', encoding='utf-8') as f:
    text = normalize_text(f.read())


sentences = [s.strip() for s in re.split(r'\. |\? |\! ', text) if len(s.strip()) > 0]

dataset_mlm = Dataset.from_dict({"text": sentences})

dataset_mlm = dataset_mlm.train_test_split(test_size=0.2, seed=42)



def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding=False)

tokenized_datasets = dataset_mlm.map(tokenize_function, batched=True, remove_columns=["text"])


tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm_probability=0.15)


# model_mlm = AutoModelForMaskedLM.from_pretrained("distilroberta-base")

model_mlm = AutoModelForMaskedLM.from_pretrained("/content/drive/MyDrive/DL_HW4/mlm_model_final")
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/DL_HW4/mlm_model",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    save_strategy="epoch",
    push_to_hub=False,
    report_to=[])


trainer = Trainer(
    model=model_mlm,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    tokenizer=tokenizer)




Map:   0%|          | 0/24790 [00:00<?, ? examples/s]

Map:   0%|          | 0/6198 [00:00<?, ? examples/s]

/tmp/ipython-input-2632270672.py:50: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [22]:

eval_before = trainer.evaluate()
print(f"ДО ОБУЧЕНИЯ: loss = {eval_before['eval_loss']:.4f}, perplexity = {math.exp(eval_before['eval_loss']):.2f}")

trainer.train()

eval_after = trainer.evaluate()
print(f"ПОСЛЕ ОБУЧЕНИЯ: loss = {eval_after['eval_loss']:.4f}, perplexity = {math.exp(eval_after['eval_loss']):.2f}")



The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


ДО ОБУЧЕНИЯ: loss = 1.3563, perplexity = 3.88


Step,Training Loss
500,1.304600
1000,1.167100
1500,1.065000
2000,0.998000
2500,0.938100
3000,0.913300
3500,0.857000
4000,0.847800
4500,0.804800
5000,0.785100


ПОСЛЕ ОБУЧЕНИЯ: loss = 0.6236, perplexity = 1.87


In [56]:
eval_before = trainer.evaluate()
print(f"ДО ОБУЧЕНИЯ: loss = {eval_before['eval_loss']:.4f}, perplexity = {math.exp(eval_before['eval_loss']):.2f}")

trainer.train()

eval_after = trainer.evaluate()
print(f"ПОСЛЕ ОБУЧЕНИЯ: loss = {eval_after['eval_loss']:.4f}, perplexity = {math.exp(eval_after['eval_loss']):.2f}")

ДО ОБУЧЕНИЯ: loss = 0.6920, perplexity = 2.00


Step,Training Loss
500,0.696300
1000,0.672800
1500,0.628200
2000,0.602000
2500,0.568500
3000,0.569800
3500,0.537700
4000,0.541900
4500,0.521100
5000,0.525500


ПОСЛЕ ОБУЧЕНИЯ: loss = 0.5364, perplexity = 1.71


In [57]:
trainer.save_model("/content/drive/MyDrive/DL_HW4/mlm_model_final2")



3. **Перетренировать классификатор из пункта 1**, но использовать претренированные веса из пункта 2
* сравнить метрики с пунктом 1

In [58]:
from transformers import AutoModel, AutoModelForSequenceClassification


base_mlm = AutoModel.from_pretrained("/content/drive/MyDrive/DL_HW4/mlm_model_final2")

model_third = AutoModelForSequenceClassification.from_pretrained(
    "/content/drive/MyDrive/DL_HW4/model_first_final2",
    num_labels=2)

model_third.roberta.load_state_dict(base_mlm.state_dict(), strict=False)


# model_third = AutoModelForSequenceClassification.from_pretrained(
#     "/content/drive/MyDrive/DL_HW4/model_third_final3",
#     num_labels=2
# )

Some weights of RobertaModel were not initialized from the model checkpoint at /content/drive/MyDrive/DL_HW4/mlm_model_final2 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


_IncompatibleKeys(missing_keys=[], unexpected_keys=['pooler.dense.weight', 'pooler.dense.bias'])

In [42]:
from transformers import TrainingArguments, Trainer
import numpy as np
import pandas as pd


In [59]:

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    conf_m = confusion_matrix(labels, predictions)
    a_score = accuracy_score(labels, predictions)
    f1_s = f1_score(labels, predictions, average="weighted")
    return {"CONFUSION_MATRIX": conf_m, "ACCURACY": a_score, "F1": f1_s}

training_args = TrainingArguments(
    output_dir="./results3",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    save_strategy="epoch",
    logging_strategy="epoch",
    push_to_hub=False,
    report_to=[]
)


trainer = Trainer(
    model=model_third,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=valid_data,
    compute_metrics=compute_metrics
)



In [44]:
trainer.train()

results3 = trainer.evaluate()

Step,Training Loss
564,0.347300
1128,0.318500
1692,0.236800
2256,0.204500
2820,0.179000


In [52]:
trainer.train()

results3 = trainer.evaluate()

Step,Training Loss
564,0.206600


In [ ]:
trainer.train()

results3 = trainer.evaluate()

Step,Training Loss
564,0.603500
1128,0.492900
1692,0.421300


In [45]:
print('МЕТРИКИ:')
print(f"Accuracy = {results3['eval_ACCURACY']}")
print(f"F1 = {results3['eval_F1']}")
print('Confusion matrix:')
print(results3['eval_CONFUSION_MATRIX'])

МЕТРИКИ:
Accuracy = 0.7686170212765957
F1 = 0.767437732890735
Confusion matrix:
[[384 163]
 [ 98 483]]


In [46]:

directory = "/content/drive/MyDrive/DL_HW4/model_third_final4"
trainer.save_model(directory)


In [54]:
import pandas as pd
import torch

data_p = "/content/drive/MyDrive/DL_HW4/submission.csv"
save_p = "/content/drive/MyDrive/DL_HW4/submission_predictions_manual_proverka.csv"

df = pd.read_csv(data_p)
texts = [normalize_text(t) for t in df['text']]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(device)

model_third.to(device).eval()
with torch.no_grad():
    df["label"] = model_third(**inputs).logits.argmax(dim=1).cpu().numpy()

df.to_csv(save_p, index=False)

